# Enhancing Real Estate Investment Decisions for Surprise Housing
## Advanced Regression Modeling & Predictive Analytics

### Executive Summary
Surprise Housing, a US-based real estate investment firm, is expanding into the Australian market. This notebook implements an end-to-end Machine Learning pipeline to predict house prices, evaluate regularization techniques (**Lasso** and **Ridge**), test tree-based ensembles (**Random Forest** and **XGBoost**), and extract key property drivers to guide strategic acquisition and valuation decisions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to sys.path
sys.path.append('..')

from generate_dataset import generate_surprise_housing_data
from src.preprocessing import preprocess_housing_data
from src.models import (
    train_and_tune_lasso,
    train_and_tune_ridge,
    train_random_forest,
    train_xgboost,
    evaluate_all_models,
    extract_feature_importance
)

sns.set_theme(style='darkgrid')
print('All libraries imported successfully.')

### Step 1: Data Exploration & Preprocessing
We inspect dataset dimensions, summary statistics, missing values, and target variable skewness (`SalePrice`). To satisfy linear model assumptions and stabilize variance, logarithmic transformation `np.log1p(SalePrice)` is applied.

In [ ]:
# Load or generate dataset
df = generate_surprise_housing_data()
print(f'Dataset dimensions: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

### Step 2: Feature Engineering & Preprocessing
We engineer impactful features:
- `TotalSF`: Sum of basement, 1st floor, and 2nd floor areas
- `HouseAge`: Age of property at year sold (`YrSold` - `YearBuilt`)
- `RemodAge`: Years since last remodel (`YrSold` - `YearRemodAdd`)
- `TotalBaths`: Weighted total bathrooms

Strict ML featurization ordering is enforced: train-test split is executed **BEFORE** scaling (`RobustScaler`) to eliminate data leakage risks.

In [ ]:
X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw, scaler, feature_names = preprocess_housing_data(df)
print(f'Training Set: {X_train.shape[0]} samples | Test Set: {X_test.shape[0]} samples')
print(f'Total Encoded Features: {len(feature_names)}')

### Step 3: Model Training & Hyperparameter Tuning
We train and tune 4 competitive models:
1. **Lasso (L1 Regularization)** with 5-fold Cross Validation grid search over $\alpha \in [10^{-4}, 10^{2}]$
2. **Ridge (L2 Regularization)** with 5-fold Cross Validation grid search over $\alpha \in [10^{-3}, 10^{3}]$
3. **Random Forest Regressor** (Ensemble Bagging)
4. **XGBoost Regressor** (Gradient Boosting)

In [ ]:
lasso, lasso_alpha, _ = train_and_tune_lasso(X_train, y_train_log)
ridge, ridge_alpha, _ = train_and_tune_ridge(X_train, y_train_log)
rf = train_random_forest(X_train, y_train_log)
xgb = train_xgboost(X_train, y_train_log)

models = {
    'Lasso Regression': lasso,
    'Ridge Regression': ridge,
    'Random Forest': rf,
    'XGBoost': xgb
}
print('Model training and hyperparameter tuning complete.')

### Step 4: Model Evaluation & Benchmark Analysis
Evaluating model predictions transformed back to original Dollar scale (`np.expm1`).

In [ ]:
results = evaluate_all_models(models, X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw)
eval_df = pd.DataFrame(results).T[['Train_R2', 'Test_R2', 'Test_MAE', 'Test_RMSE', 'Test_MAPE']]
eval_df.columns = ['Train R2', 'Test R2', 'Test MAE ($)', 'Test RMSE ($)', 'Test MAPE (%)']
eval_df

### Step 5: Feature Importance & Business Insights for Investment Decisions
Lasso regression shrinks non-essential feature coefficients to exact zeros, leaving only the most significant drivers of property valuation.

In [ ]:
coefs = extract_feature_importance(lasso, feature_names)
top_pos = coefs.sort_values(ascending=False).head(10)
top_neg = coefs.sort_values(ascending=True).head(5)

print('Top Positive Price Drivers:')
print(top_pos)
print('\nTop Negative Price Drivers:')
print(top_neg)

### Conclusion & Strategic Recommendations
1. **Primary Price Drivers**: Total living square footage (`TotalSF`, `GrLivArea`), Overall Quality (`OverallQual`), and prime neighborhoods (`NridgHt`, `NoRidge`, `StoneBr`) yield the strongest positive ROI multipliers.
2. **Depreciating Factors**: Property age (`HouseAge`) and poor condition ratings decrease property market value exponentially.
3. **Model Selection**: Lasso Regression offers robust interpretability for feature elimination with high test $R^2$, while XGBoost provides maximum predictive precision for automated valuation.